# 8. Find things with YOLO

Use a [YOLO](https://docs.ultralytics.com/) model to find people, dogs, cars and other things in your Cameras.

YOLO comes with the notebook tools (`googlenestcam[notebooks]`), from the `ultralytics` package. The first run downloads a small model file (about 5 MB). Note: `ultralytics` uses the AGPL-3.0 licence, so check it fits your project before you ship it.

Log in first with [1. Quick start](01-quick-start.ipynb).

In [ ]:
from collections import Counter

from IPython.display import display
from PIL import Image
from ultralytics import YOLO

import googlenestcam as gnc

model = YOLO("yolo11n.pt")
cameras = gnc.list_cameras()
print([cam.name for cam in cameras])

## Snapshots from every Camera

Take one Snapshot from each Camera and ask the model what it sees. `frame.to_pil()` gives an RGB picture, which YOLO reads as is. `result.plot()` draws the boxes and gives back a BGR array, so `[..., ::-1]` swaps it back to RGB.

In [ ]:
for cam in cameras:
    frame = cam.snapshot(size=640)
    result = model(frame.to_pil(), verbose=False)[0]
    found = Counter(result.names[int(c)] for c in result.boxes.cls)
    print(cam.name, dict(found))
    display(Image.fromarray(result.plot()[..., ::-1]))

## Each thing found

Every box has a name, a confidence from 0 to 1, and its corners in pixels (left, top, right, bottom). `conf=0.5` drops guesses the model is less sure about.

In [ ]:
result = model(frame.to_pil(), conf=0.5, verbose=False)[0]
for box in result.boxes:
    name = result.names[int(box.cls)]
    corners = [round(v) for v in box.xyxy[0].tolist()]
    print(f"{name}: {float(box.conf):.2f} at {corners}")

## Boxes on the live Stream for 30 seconds

Run the model on each Frame and show the picture with boxes. The Stream keeps only the newest Frame, so if the model is slow it skips old Frames instead of falling behind. Stop the cell at any time with the stop button.

In [ ]:
import time

with cameras[0].stream(size=480, audio=False) as stream:
    view = display(None, display_id=True)
    end = time.monotonic() + 30
    for frame in stream.frames():
        result = model(frame.to_pil(), verbose=False)[0]
        view.update(Image.fromarray(result.plot()[..., ::-1]))
        if time.monotonic() > end:
            break